Paths and reusable definitions were refactored. Stored outputs below are historical, not a fresh execution. See `docs/WORKFLOW.md` at the repository root before rerunning; several stages overwrite results or append Optuna trials.


In [ ]:
# Locate this checkout from the repository root or any notebook subdirectory.
from pathlib import Path
import sys

_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
              if (p / "src" / "steam_recommender" / "paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter inside the repository checkout.")
sys.path.insert(0, str(_root / "src"))
from steam_recommender import paths as _paths
if Path(_paths.__file__).resolve() != (_root / "src/steam_recommender/paths.py").resolve():
    raise RuntimeError("A different checkout is cached; restart the kernel.")
from steam_recommender.paths import project_root, project_path


In [1]:
# =========================
# BPR EXPERIMENT
# Bayesian Personalized Ranking as binary implicit-feedback baseline
# =========================

import os

os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import sys
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from tqdm.auto import tqdm
from threadpoolctl import threadpool_limits

threadpool_limits(1, "blas")


# =========================
# PATHS
# =========================


data_dir = os.path.join(str(project_path("data/processed")))
optuna_results_dir = os.path.join(str(project_path("results/optuna")))
final_results_dir = os.path.join(str(project_path("results/final")))

os.makedirs(optuna_results_dir, exist_ok=True)
os.makedirs(final_results_dir, exist_ok=True)

# =========================
# IMPORTS FROM EXISTING PIPELINE
# =========================

from steam_recommender.splitting import random_user_split_train_val
from steam_recommender.metrics import precision_at_k, recall_at_k, ndcg_at_k, coverage_at_k, delta_gap, diversity_at_k

from implicit.bpr import BayesianPersonalizedRanking


# =========================
# OUTPUT PATHS
# =========================

# =========================
# BPR OUTPUT PATHS
# =========================

bpr_results_dir = os.path.join(str(project_path("results/bpr")))
bpr_optuna_dir = os.path.join(bpr_results_dir, "optuna")
bpr_final_dir = os.path.join(bpr_results_dir, "final")

os.makedirs(bpr_results_dir, exist_ok=True)
os.makedirs(bpr_optuna_dir, exist_ok=True)
os.makedirs(bpr_final_dir, exist_ok=True)

bpr_storage_path = os.path.join(
    bpr_optuna_dir,
    "bpr_studies.db"
)

bpr_optuna_csv_path = os.path.join(
    bpr_optuna_dir,
    "optuna_bpr_results.csv"
)

bpr_test_metrics_path = os.path.join(
    bpr_final_dir,
    "bpr_test_metrics_by_seed.csv"
)

bpr_per_user_ndcg_path = os.path.join(
    bpr_final_dir,
    "bpr_per_user_ndcg.csv"
)

In [2]:
# =========================
# LOAD DATA
# =========================

train_val_df = pd.read_csv(
    os.path.join(data_dir, "users_items_filtered_train_val.csv")
)

test_df = pd.read_csv(
    os.path.join(data_dir, "users_items_filtered_test.csv")
)

for df in [train_val_df, test_df]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

print("train_val_df:", train_val_df.shape)
print("test_df:", test_df.shape)

C:\Users\misie\AppData\Local\Temp\ipykernel_7124\866791025.py:5: DtypeWarning: Columns (0: user_id) have mixed types. Specify dtype option on import or set low_memory=False.
  train_val_df = pd.read_csv(


train_val_df: (4591379, 4)
test_df: (541231, 4)


In [3]:
# =========================
# BPR PIPELINE
# =========================

from steam_recommender.bpr import BPRPipeline


In [4]:
# =========================
# TRAIN / VAL SPLIT
# same as ALS Optuna
# =========================

train_df, val_df = random_user_split_train_val(
    train_val_df,
    val_frac=0.1,
    seed=42,
)

print("train_df:", train_df.shape)
print("val_df:", val_df.shape)

train_df: (4104618, 4)
val_df: (486761, 4)


In [5]:
# =========================
# OPTUNA — BPR
# tuned search space after initial exploratory run
# =========================

import optuna

from optuna.samplers import TPESampler
from optuna.pruners import HyperbandPruner
from optuna.storages import RDBStorage


# =========================
# STORAGE
# =========================

storage_path = bpr_storage_path

storage = RDBStorage(
    url=f"sqlite:///{storage_path}",
    engine_kwargs={
        "connect_args": {
            "timeout": 60,
        }
    },
)


# =========================
# OBJECTIVE
# =========================

def objective_bpr(trial):
    """
    Jeden trial = jedna konfiguracja BPR.

    BPR traktujemy jako binarny model implicit-feedback:
    - obserwowana interakcja user-item = pozytyw
    - brak interakcji = kandydat negatywny w samplingu BPR

    Optymalizujemy NDCG@10 na tym samym validation split co ALS.
    """

    # Pierwszy test pokazał najlepsze wyniki raczej dla małych/średnich faktorów.
    factors = trial.suggest_int(
        "factors",
        16,
        96,
        step=8,
    )

    # Pierwszy test wskazywał okolice 0.003–0.009.
    # Dajemy trochę szerszy, ale skoncentrowany zakres.
    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-3,
        2e-2,
        log=True,
    )

    # Pierwszy test wskazywał okolice 0.001–0.003.
    # Dajemy szerszy zakres wokół tego regionu.
    regularization = trial.suggest_float(
        "regularization",
        3e-4,
        2e-2,
        log=True,
    )

    # Dajemy większy zakres, żeby sprawdzić czy BPR był niedotrenowany.
    iterations = trial.suggest_categorical(
        "iterations",
        [50, 100, 150, 200, 300],
    )

    pipeline = BPRPipeline(
        factors=factors,
        iterations=iterations,
        learning_rate=learning_rate,
        regularization=regularization,
        random_state=0,
        num_threads=8,
        verify_negative_samples=True,
    )

    pipeline.fit(train_df, val_df)

    ndcg = pipeline.evaluate_fast(K=10)

    if np.isnan(ndcg):
        return 0.0

    return ndcg


# =========================
# CREATE NEW STUDY
# =========================

study_bpr = optuna.create_study(
    study_name="bpr_binary_tuned",
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=HyperbandPruner(),
    storage=storage,
    load_if_exists=False,
)


# =========================
# OPTIONAL: ENQUEUE PREVIOUSLY GOOD REGIONS
# =========================
# Te konfiguracje są blisko dobrych wyników z pierwszego szybkiego testu.
# Optuna sprawdzi je na początku.

study_bpr.enqueue_trial({
    "factors": 24,
    "learning_rate": 0.004,
    "regularization": 0.0028,
    "iterations": 100,
})

study_bpr.enqueue_trial({
    "factors": 40,
    "learning_rate": 0.0055,
    "regularization": 0.0021,
    "iterations": 100,
})

study_bpr.enqueue_trial({
    "factors": 32,
    "learning_rate": 0.007,
    "regularization": 0.0017,
    "iterations": 150,
})

study_bpr.enqueue_trial({
    "factors": 24,
    "learning_rate": 0.004,
    "regularization": 0.0028,
    "iterations": 200,
})


# =========================
# RUN OPTUNA
# =========================

study_bpr.optimize(
    objective_bpr,
    n_trials=150,
    n_jobs=1,
    show_progress_bar=True,
)


# =========================
# RESULTS
# =========================

print("Best BPR NDCG:", study_bpr.best_value)
print("Best BPR params:")
print(study_bpr.best_params)

bpr_trials_df = study_bpr.trials_dataframe()

bpr_results_df = (
    bpr_trials_df
    .sort_values("value", ascending=False)
    .reset_index(drop=True)
)

display(
    bpr_results_df[
        [
            "number",
            "value",
            "params_factors",
            "params_learning_rate",
            "params_regularization",
            "params_iterations",
            "state",
        ]
    ].head(30)
)

bpr_results_df.to_csv(bpr_optuna_csv_path, index=False)

print(f"Saved BPR Optuna results to: {bpr_optuna_csv_path}")
print(f"BPR Optuna storage: {storage_path}")

[I 2026-05-19 19:12:16,235] A new study created in RDB with name: bpr_binary_tuned


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-19 19:12:30,922] Trial 0 finished with value: 0.19033405367290115 and parameters: {'factors': 24, 'learning_rate': 0.004, 'regularization': 0.0028, 'iterations': 100}. Best is trial 0 with value: 0.19033405367290115.


  0%|          | 0/100 [00:00<?, ?it/s]

...

[I 2026-05-19 20:26:28,703] Trial 148 finished with value: 0.20941418053701952 and parameters: {'factors': 96, 'learning_rate': 0.0022761664304105722, 'regularization': 0.0035072657978117753, 'iterations': 150}. Best is trial 132 with value: 0.2118049701017791.


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-05-19 20:26:58,814] Trial 149 finished with value: 0.20709626101921752 and parameters: {'factors': 96, 'learning_rate': 0.0017144327183312787, 'regularization': 0.002075523967842273, 'iterations': 150}. Best is trial 132 with value: 0.2118049701017791.
Best BPR NDCG: 0.2118049701017791
Best BPR params:
{'factors': 96, 'learning_rate': 0.0017602669790432763, 'regularization': 0.0032374923658688454, 'iterations': 150}


,number,value,params_factors,params_learning_rate,params_regularization,params_iterations,state
0,132,0.211805,96,0.001760,0.003237,150,COMPLETE
1,130,0.211690,96,0.001753,0.003389,150,COMPLETE
2,102,0.211646,96,0.001845,0.003864,150,COMPLETE
3,127,0.211613,96,0.001684,0.003479,150,COMPLETE
4,131,0.211538,96,0.001734,0.003346,150,COMPLETE
5,103,0.211534,96,0.001874,0.003528,150,COMPLETE
6,94,0.211534,96,0.001720,0.003565,150,COMPLETE
7,111,0.211446,96,0.001874,0.003879,150,COMPLETE
8,141,0.211441,96,0.001766,0.003148,150,COMPLETE
9,119,0.211366,96,0.001883,0.003472,150,COMPLETE


Saved BPR Optuna results to: D:\Studiowe\Licencjat\modele\bpr_results\optuna\optuna_bpr_results.csv
BPR Optuna storage: D:\Studiowe\Licencjat\modele\bpr_results\optuna\bpr_studies.db


In [6]:
# =========================
# FINAL TEST — BPR
# =========================

best_bpr_params = study_bpr.best_params

K = 10
seeds = [0, 1, 2, 3, 4]

metric_rows = []
per_user_rows = []

for seed in tqdm(seeds, desc="BPR final test"):
    pipeline = BPRPipeline(
        factors=int(best_bpr_params["factors"]),
        iterations=int(best_bpr_params["iterations"]),
        learning_rate=float(best_bpr_params["learning_rate"]),
        regularization=float(best_bpr_params["regularization"]),
        random_state=seed,
        num_threads=8,
        verify_negative_samples=True,
    )

    results, recommendations, user_ndcg = pipeline.refit_full_and_test(
        train_df=train_val_df,
        test_df=test_df,
        K=K,
        return_per_user=True,
        verbose=False,
    )

    metric_row = {
        "model": "BPR",
        "function": "bpr_binary",
        "function_label": "BPR",
        "seed": seed,
        "factors": int(best_bpr_params["factors"]),
        "iterations": int(best_bpr_params["iterations"]),
        "learning_rate": float(best_bpr_params["learning_rate"]),
        "regularization": float(best_bpr_params["regularization"]),
    }

    metric_row.update(results)
    metric_rows.append(metric_row)

    for user_id, ndcg_value in user_ndcg.items():
        per_user_rows.append({
            "model": "BPR",
            "function": "bpr_binary",
            "function_label": "BPR",
            "seed": seed,
            "user_id": user_id,
            f"ndcg@{K}": ndcg_value,
        })


bpr_test_metrics_df = pd.DataFrame(metric_rows)
bpr_per_user_ndcg_df = pd.DataFrame(per_user_rows)

display(bpr_test_metrics_df)
display(bpr_per_user_ndcg_df.head())

bpr_test_metrics_df.to_csv(bpr_test_metrics_path, index=False)
bpr_per_user_ndcg_df.to_csv(bpr_per_user_ndcg_path, index=False)

print(f"Saved BPR test metrics to: {bpr_test_metrics_path}")
print(f"Saved BPR per-user NDCG to: {bpr_per_user_ndcg_path}")

BPR final test:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

,model,function,function_label,seed,factors,iterations,learning_rate,regularization,precision@10,recall@10,ndcg@10,coverage@10,delta_gap@10,diversity@10,users_evaluated
0,BPR,bpr_binary,BPR,0,96,150,0.00176,0.003237,0.132896,0.261243,0.269803,0.305827,-0.079945,0.549656,62944
1,BPR,bpr_binary,BPR,1,96,150,0.00176,0.003237,0.132508,0.260874,0.270121,0.303653,-0.086190,0.550582,62944
2,BPR,bpr_binary,BPR,2,96,150,0.00176,0.003237,0.133287,0.262033,0.271325,0.303979,-0.078522,0.549720,62944
3,BPR,bpr_binary,BPR,3,96,150,0.00176,0.003237,0.132893,0.261749,0.270106,0.303870,-0.071132,0.550916,62944
4,BPR,bpr_binary,BPR,4,96,150,0.00176,0.003237,0.132673,0.261104,0.269937,0.302348,-0.078663,0.548628,62944


,model,function,function_label,seed,user_id,ndcg@10
0,BPR,bpr_binary,BPR,0,--000--,0.100867
1,BPR,bpr_binary,BPR,0,--ace--,0.000000
2,BPR,bpr_binary,BPR,0,--ionex--,0.148041
3,BPR,bpr_binary,BPR,0,-2SV-vuLB-Kg,0.079457
4,BPR,bpr_binary,BPR,0,-404PageNotFound-,0.220092


Saved BPR test metrics to: D:\Studiowe\Licencjat\modele\bpr_results\final\bpr_test_metrics_by_seed.csv
Saved BPR per-user NDCG to: D:\Studiowe\Licencjat\modele\bpr_results\final\bpr_per_user_ndcg.csv


In [7]:

# =========================
# BPR SUMMARY
# =========================

metric_cols = [
    f"precision@{K}",
    f"recall@{K}",
    f"ndcg@{K}",
    f"coverage@{K}",
    f"delta_gap@{K}",
    f"diversity@{K}",
    "users_evaluated",
]

summary = {}

for metric in metric_cols:
    summary[f"{metric}_mean"] = bpr_test_metrics_df[metric].mean()
    summary[f"{metric}_std"] = bpr_test_metrics_df[metric].std(ddof=1)

bpr_summary_df = pd.DataFrame([summary])

display(bpr_summary_df)

print("BPR summary:")
for metric in metric_cols:
    mean = summary[f"{metric}_mean"]
    std = summary[f"{metric}_std"]

    if metric == "users_evaluated":
        print(f"{metric}: {mean:.0f} ± {std:.0f}")
    else:
        print(f"{metric}: {mean:.4f} ± {std:.4f}")

,precision@10_mean,precision@10_std,recall@10_mean,recall@10_std,ndcg@10_mean,ndcg@10_std,coverage@10_mean,coverage@10_std,delta_gap@10_mean,delta_gap@10_std,diversity@10_mean,diversity@10_std,users_evaluated_mean,users_evaluated_std
0,0.132851,0.000293,0.261401,0.000477,0.270258,0.00061,0.303936,0.001243,-0.078891,0.005357,0.5499,0.000896,62944.0,0.0


BPR summary:
precision@10: 0.1329 ± 0.0003
recall@10: 0.2614 ± 0.0005
ndcg@10: 0.2703 ± 0.0006
coverage@10: 0.3039 ± 0.0012
delta_gap@10: -0.0789 ± 0.0054
diversity@10: 0.5499 ± 0.0009
users_evaluated: 62944 ± 0


In [3]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(str(project_path("results/bpr/final")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bpr_summary_pretty = pd.DataFrame([
    {
        "Model": "BPR",
        "precision@10": "0.1329 ± 0.0003",
        "recall@10": "0.2614 ± 0.0005",
        "ndcg@10": "0.2703 ± 0.0006",
        "coverage@10": "0.3039 ± 0.0012",
        "delta_gap@10": "-0.0789 ± 0.0054",
        "diversity@10": "0.5499 ± 0.0009",
        "users_evaluated": "62944 ± 0",
    }
])

output_path = OUTPUT_DIR / "bpr_test_metrics_summary_pretty.csv"

bpr_summary_pretty.to_csv(output_path, index=False, encoding="utf-8")

print(f"Saved to: {output_path}")
display(bpr_summary_pretty)

Saved to: D:\Studiowe\Licencjat\modele\bpr_results\final\bpr_test_metrics_summary_pretty.csv


,Model,precision@10,recall@10,ndcg@10,coverage@10,delta_gap@10,diversity@10,users_evaluated
0,BPR,0.1329 ± 0.0003,0.2614 ± 0.0005,0.2703 ± 0.0006,0.3039 ± 0.0012,-0.0789 ± 0.0054,0.5499 ± 0.0009,62944 ± 0
